# 🤖 Notebook 03 — Forecasting Models
### Walmart Retail Sales Demand Forecasting

---

## 🎯 Goal

Build and compare three forecasting approaches — starting from the simplest possible guess and working up to a proper time-series model. Along the way, we'll catch and fix a critical data-split bug that would have made all results invalid.

**Models we'll test:**

| # | Model | Core Idea |
|---|---|---|
| 1 | Last week's sales | "Next week = this week" |
| 2 | 4-week rolling average | Smooth out recent weeks |
| 3 | Prophet (per-store) | Learn yearly seasonality properly |


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('../data/walmart_sales.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

print(f'Loaded {len(df):,} rows | {df["Store"].nunique()} stores | {df["Date"].nunique()} weeks')

## 2. Train / Test Split — and a Critical Bug to Avoid

We need to split the data into:
- **Training set** — what the model learns from (first 80% of weeks)
- **Test set** — what we use to measure accuracy (last 20% of weeks)

### ⚠️ The Bug: Global Split

The most natural way to split looks like this:

```python
split = int(len(df) * 0.8)
train = df.iloc[:split]   # first 80% of rows
test  = df.iloc[split:]   # last 20% of rows
```

**This is wrong.** Because the data is ordered by Store first (Store 1 all rows, then Store 2, etc.), this split accidentally puts Store 1–36 entirely in training and Stores 37–45 entirely in test. Prophet would be asked to predict stores it has *never seen* — and would fall back to the global average (~$1M), while those stores actually sell ~$300K/week. Error: **$700K+ per prediction.**

### ✅ The Fix: Per-Store Split

Apply the 80/20 split **within each store independently**. Every store contributes both training weeks and test weeks.


In [ ]:
# Build train/test by splitting each store's timeline 80/20
train_list, test_list = [], []

for store_id in sorted(df['Store'].unique()):
    store_df = df[df['Store'] == store_id].sort_values('Date').reset_index(drop=True)
    split_point = int(len(store_df) * 0.8)
    train_list.append(store_df.iloc[:split_point])
    test_list.append(store_df.iloc[split_point:])

train = pd.concat(train_list).reset_index(drop=True)
test  = pd.concat(test_list).reset_index(drop=True)

print(f'Train : {len(train):,} rows | {train["Store"].nunique()} stores | up to {train["Date"].max().date()}')
print(f'Test  : {len(test):,}  rows | {test["Store"].nunique()} stores | from {test["Date"].min().date()}')
print(f'\n✅ Every store has both train and test data.')

## 3. Baseline Model 1 — Last Week's Sales

The simplest possible forecast: whatever each store sold in its last training week, predict that for every test week.

This is the *floor* — any serious model should beat this.


In [ ]:
# For each store, take the last sales value from training and use it as the prediction
last_week_per_store = train.groupby('Store')['Weekly_Sales'].last()
test['pred_last_week'] = test['Store'].map(last_week_per_store)

print('Last week predictions assigned to all test rows.')
print(test[['Store', 'Date', 'Weekly_Sales', 'pred_last_week']].head(5))

## 4. Baseline Model 2 — 4-Week Rolling Average

A slight improvement: average the last 4 weeks of training data per store. This smooths out any single-week spike or dip that might make last week's number misleading.


In [ ]:
# Compute the rolling 4-week average at the end of training for each store
rolling_per_store = (
    train
    .groupby('Store')['Weekly_Sales']
    .apply(lambda x: x.rolling(window=4).mean().iloc[-1])
)
test['pred_rolling_avg'] = test['Store'].map(rolling_per_store)

print('4-week rolling average predictions assigned.')

## 5. Evaluate the Baselines

We use three metrics:

| Metric | What it means |
|---|---|
| **MAE** (Mean Absolute Error) | Average dollar error per prediction |
| **RMSE** (Root Mean Squared Error) | Similar to MAE but penalises big errors more |
| **MAPE** (Mean Absolute % Error) | Error as a percentage of actual sales — easiest to interpret |


In [ ]:
def evaluate(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = (np.abs(actual - predicted) / actual).mean() * 100
    print(f'{model_name:<25}  MAE: ${mae:>10,.0f}   RMSE: ${rmse:>10,.0f}   MAPE: {mape:.1f}%')

print(f'{"Model":<25}  {"MAE":>13}   {"RMSE":>13}   {"MAPE":>6}')
print('-' * 72)
evaluate(test['Weekly_Sales'], test['pred_last_week'],   'Last Week Baseline')
evaluate(test['Weekly_Sales'], test['pred_rolling_avg'], 'Rolling Avg (4-wk)')

In [ ]:
# Plot baseline predictions vs actual for one representative store
store_to_plot = 1
s = test[test['Store'] == store_to_plot].sort_values('Date')

plt.figure(figsize=(14, 5))
plt.plot(s['Date'], s['Weekly_Sales'],       label='Actual sales',          color='steelblue', linewidth=2)
plt.plot(s['Date'], s['pred_last_week'],     label='Last week baseline',    color='coral',    linestyle='--')
plt.plot(s['Date'], s['pred_rolling_avg'],   label='Rolling avg (4-wk)',    color='orange',   linestyle='--')
plt.title(f'Baseline Forecasts vs Actual — Store {store_to_plot}', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

### 💡 Key insight

Both baselines produce flat lines — they predict the same number every week regardless of what time of year it is. They completely miss holiday spikes, which is exactly where getting the prediction right matters most. This motivates moving to a seasonality-aware model.


## 6. Model 3 — Prophet (Per-Store Seasonal Model)

**What is Prophet?**  
Prophet is a time-series forecasting library built by Facebook. It decomposes sales into three components:

- **Trend** — is the overall level going up or down over time?
- **Seasonality** — what's the typical pattern within a year? (e.g., December is always busy)
- **Residuals** — what's left over after removing trend and seasonality?

**Why per-store?**  
From the EDA, we know that each store has very different sales volumes. A single global model would average across all stores and do a poor job for each individual one. Training one Prophet model per store gives each store its own trend and seasonal pattern.

**Configuration choices:**
- `yearly_seasonality=True` — we have 2.5 years of data, so Prophet can learn annual patterns
- `weekly_seasonality=False` — our data is already weekly, so there's no within-week pattern to learn
- `seasonality_mode='multiplicative'` — in retail, holiday spikes are proportional to a store's baseline sales (a big store has a bigger absolute spike than a small store)


In [ ]:
# Prophet requires columns named 'ds' (date) and 'y' (value)
# Build one training dataframe per store
store_prophet_data = {}

for store_id in sorted(train['Store'].unique()):
    store_df = (
        train[train['Store'] == store_id][['Date', 'Weekly_Sales']]
        .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
        .copy()
    )
    store_prophet_data[store_id] = store_df

print(f'Prophet training data ready for {len(store_prophet_data)} stores.')
print(f'Store 1 example — {len(store_prophet_data[1])} weeks, avg ${store_prophet_data[1]["y"].mean():,.0f}/week')

In [ ]:
from prophet import Prophet

store_models = {}

for store_id, prophet_df in store_prophet_data.items():
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative'  # spikes scale with store size
    )
    model.fit(prophet_df)
    store_models[store_id] = model

print(f'✅ Trained {len(store_models)} Prophet models — one per store.')

In [ ]:
# Generate predictions for each store's test period
pred_rows = []

for store_id, model in store_models.items():
    test_store = test[test['Store'] == store_id].copy().reset_index(drop=True)
    n_weeks    = len(test_store)

    # Ask Prophet to forecast n_weeks beyond the training period
    future   = model.make_future_dataframe(periods=n_weeks, freq='W')
    forecast = model.predict(future)

    # Take only the forecasted test weeks, clip negatives (sales can't be < 0)
    predictions = forecast.iloc[-n_weeks:]['yhat'].clip(lower=0).values
    test_store['pred_prophet'] = predictions
    pred_rows.append(test_store)

# Combine back into one dataframe
test = pd.concat(pred_rows).sort_values(['Store', 'Date']).reset_index(drop=True)

print(f'✅ Prophet predictions generated for all {test["Store"].nunique()} stores.')
print(test[['Store', 'Date', 'Weekly_Sales', 'pred_prophet']].head())

## 7. Evaluate All Three Models

In [ ]:
print(f'{'Model':<25}  {'MAE':>13}   {'RMSE':>13}   {'MAPE':>6}')
print('-' * 72)
evaluate(test['Weekly_Sales'], test['pred_last_week'],   'Last Week Baseline')
evaluate(test['Weekly_Sales'], test['pred_rolling_avg'], 'Rolling Avg (4-wk)')
evaluate(test['Weekly_Sales'], test['pred_prophet'],     'Prophet (per-store)')

In [ ]:
# Visual comparison for Store 1
store_to_plot = 1
s = test[test['Store'] == store_to_plot].sort_values('Date')

plt.figure(figsize=(14, 5))
plt.plot(s['Date'], s['Weekly_Sales'],     label='Actual sales',     color='steelblue', linewidth=2)
plt.plot(s['Date'], s['pred_rolling_avg'], label='Rolling avg',      color='orange',   linestyle='--', alpha=0.7)
plt.plot(s['Date'], s['pred_prophet'],     label='Prophet forecast', color='green',    linewidth=1.5)
plt.title(f'Prophet vs Rolling Avg vs Actual — Store {store_to_plot}', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Weekly Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

## ✅ Results Summary

| Model | MAE | MAPE | What it misses |
|---|---|---|---|
| Last Week Baseline | ~$311K | ~27% | Everything — flat prediction |
| Rolling Avg (4-wk) | ~$244K | ~21% | Holiday spikes |
| **Prophet (per-store)** | **~$56K** | **~5.4%** | ✅ Captures seasonality |

**Prophet reduces error by ~80% compared to the rolling average and ~10× compared to the last-week baseline.**

The key reason: Prophet explicitly models the yearly seasonal pattern — it *knows* that December is different from July, even before seeing a single test week.

**Next:** Notebook 04 translates these accuracy improvements into real business dollars — how much does better forecasting actually save?
